### Ingestion del archivo "language"

In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
v_environment

'production'

In [0]:
%run "../Includes/configuration"

In [0]:
bronze_folder_path

'abfss://bronze@moviehistory310785.dfs.core.windows.net'

In [0]:
silver_folder_path

'abfss://silver@moviehistory310785.dfs.core.windows.net'

In [0]:
gold_folder_path

'abfss://gold@moviehistory310785.dfs.core.windows.net'

In [0]:
%run "../Includes/common_functions"

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_schema = StructType(fields = [
    StructField('LanguageId', IntegerType(), False),
    StructField('LanguageCode', StringType(), True),
    StructField('LanguageName', StringType(), True)
])

In [0]:
%fs
ls abfss://bronze@moviehistory310785.dfs.core.windows.net/

path,name,size,modificationTime
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-16/,2024-12-16/,0,0
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-23/,2024-12-23/,0,0
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-30/,2024-12-30/,0,0


In [0]:
language_df = spark.read \
              .option("header", True) \
              .schema(language_schema) \
              .csv(f"{bronze_folder_path}/{v_file_date}/language.csv")

#### Paso 2 - Seleccionar solo las columnas "requeridas"

In [0]:
from pyspark.sql.functions import col

In [0]:
language_selected_df = language_df.select(col("LanguageId"), col("LanguageName"))

#### Paso 3 - Cambiar el nombre de las columnas segun lo "requerido"

In [0]:
language_renamed_df = language_selected_df.withColumnsRenamed({"LanguageId": "Language_Id", "LanguageName": "Language_Name"})

#### Paso 4 - Agregar la columna "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
language_final_df = add_ingestion_date(language_renamed_df) \
                    .withColumn("env", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))

In [0]:
display(language_final_df)

Language_Id,Language_Name,ingestion_date,env,file_date
24574,English,2026-09-10T20:04:02.201745Z,production,2024-12-30
24575,svenska,2026-09-10T20:04:02.201745Z,production,2024-12-30
24576,Deutsch,2026-09-10T20:04:02.201745Z,production,2024-12-30
24577,unknown,2026-09-10T20:04:02.201745Z,production,2024-12-30
24578,Nihongo,2026-09-10T20:04:02.201745Z,production,2024-12-30
24579,Français,2026-09-10T20:04:02.201745Z,production,2024-12-30
24580,Español,2026-09-10T20:04:02.201745Z,production,2024-12-30
24581,al-?arabiyyah,2026-09-10T20:04:02.201745Z,production,2024-12-30
24582,Latin,2026-09-10T20:04:02.201745Z,production,2024-12-30
24583,Khémôrôphéasa,2026-09-10T20:04:02.201745Z,production,2024-12-30


#### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
language_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
display(spark.read.table("movie_silver.languages"))

Language_Id,Language_Name,ingestion_date,env,file_date
24574,English,2026-09-10T20:04:03.363772Z,production,2024-12-30
24575,svenska,2026-09-10T20:04:03.363772Z,production,2024-12-30
24576,Deutsch,2026-09-10T20:04:03.363772Z,production,2024-12-30
24577,unknown,2026-09-10T20:04:03.363772Z,production,2024-12-30
24578,Nihongo,2026-09-10T20:04:03.363772Z,production,2024-12-30
24579,Français,2026-09-10T20:04:03.363772Z,production,2024-12-30
24580,Español,2026-09-10T20:04:03.363772Z,production,2024-12-30
24581,al-?arabiyyah,2026-09-10T20:04:03.363772Z,production,2024-12-30
24582,Latin,2026-09-10T20:04:03.363772Z,production,2024-12-30
24583,Khémôrôphéasa,2026-09-10T20:04:03.363772Z,production,2024-12-30


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.languages
GROUP BY file_date

file_date,count(1)
2024-12-30,88


In [0]:
%sql
SELECT * FROM movie_silver.languages

Language_Id,Language_Name,ingestion_date,env,file_date
24574,English,2026-09-10T20:04:03.363772Z,production,2024-12-30
24575,svenska,2026-09-10T20:04:03.363772Z,production,2024-12-30
24576,Deutsch,2026-09-10T20:04:03.363772Z,production,2024-12-30
24577,unknown,2026-09-10T20:04:03.363772Z,production,2024-12-30
24578,Nihongo,2026-09-10T20:04:03.363772Z,production,2024-12-30
24579,Français,2026-09-10T20:04:03.363772Z,production,2024-12-30
24580,Español,2026-09-10T20:04:03.363772Z,production,2024-12-30
24581,al-?arabiyyah,2026-09-10T20:04:03.363772Z,production,2024-12-30
24582,Latin,2026-09-10T20:04:03.363772Z,production,2024-12-30
24583,Khémôrôphéasa,2026-09-10T20:04:03.363772Z,production,2024-12-30


In [0]:
dbutils.notebook.exit("Success")